# 005 Wire a Skill into a Mini Workflow

这是第五课：把 Skill 的几个部分接成一个完整小流程。

学习目标：

1. 理解 Skill 的多个部分什么时候开始真正协同工作
2. 学会把“标准化输入”和“构建查询”拆成两个小脚本
3. 学会让 Skill 从“有脚本”进化到“有小流程”
4. 用真实天气 Skill 形成第一条完整执行链路

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

到第四课为止，这份天气 Skill 已经有了第一个脚本：

- `normalize_location.py`

但它还只是一个单点动作。

现在我们要继续推进一步：

1. 标准化地点
2. 构建 `wttr.in` 查询 URL
3. 形成可直接用于查询的结果

这就是一个最小但完整的 Skill workflow integration。


## 先看当前真实目录

这次我们继续扩展已有 Skill，而不是新造示例。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
├── scripts
│   ├── __pycache__
│   │   └── normalize_location.cpython-310.pyc
│   ├── build_wttr_query.py
│   └── normalize_location.py
└── SKILL.md


## 先看 `SKILL.md` 的变化

这次你应该重点观察：

1. workflow 里增加了“Build a stable weather query string”
2. `Scripts` 小节里增加了 `build_wttr_query.py`

这意味着这份 Skill 已经不只是“有脚本”，而是开始描述一个完整小流程了。


In [3]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Build a stable weather query string before calling the external service.
5. Use `wttr.in` as the primary source.
6. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
7. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
8. Do not guess when weather data is unavailable.

## References

- Read `referenc

## 为什么这里需要第二个脚本

地点标准化解决的是“输入稳定性”。

但标准化之后，仍然还有一个重复动作：

- 把用户输入转成最终可查询的 `wttr.in` URL

这个动作也满足脚本化条件：

1. 会重复出现
2. 输入输出边界清晰
3. 结果应该完全可预测
4. 适合复用


In [4]:
why_second_script = {
    'repeated_action': True,
    'clear_input_output': True,
    'deterministic_result': True,
    'workflow_integration_value': True,
}

from pprint import pprint
pprint(why_second_script)


{'clear_input_output': True,
 'deterministic_result': True,
 'repeated_action': True,
 'workflow_integration_value': True}


## 读真实脚本 `build_wttr_query.py`

这个脚本的职责很单一：

- 接收原始地点输入
- 调用 `normalize_location()`
- 根据 mode 和 units 生成最终 URL

它不负责真的发起网络请求，只负责把查询构建出来。

这是一个很好的边界设计。


In [5]:
print((skill_root / 'scripts' / 'build_wttr_query.py').read_text(encoding='utf-8'))


#!/usr/bin/env python3
import argparse

from normalize_location import normalize_location


MODE_TO_FORMAT = {
    "current": "3",
    "compact": "%l:+%c+%t+%h+%w",
    "full": "T",
}


def build_wttr_query(location: str, mode: str = "compact", units: str = "metric") -> str:
    normalized = normalize_location(location)
    format_part = MODE_TO_FORMAT[mode]
    units_suffix = "m" if units == "metric" else "u"
    return f"https://wttr.in/{normalized}?{units_suffix}&format={format_part}"


def main() -> int:
    parser = argparse.ArgumentParser(description="Build a wttr.in query URL from a user-provided location.")
    parser.add_argument("location", help="Raw location input from the user")
    parser.add_argument("--mode", choices=sorted(MODE_TO_FORMAT.keys()), default="compact")
    parser.add_argument("--units", choices=["metric", "uscs"], default="metric")
    args = parser.parse_args()

    print(build_wttr_query(args.location, mode=args.mode, units=args.units))
    return 0


if 

## 直接跑几个例子

正式开发里，脚本要用真实例子验证。


In [6]:
import subprocess

examples = [
    ['python', '.agents/skills/weather-query-assistant/scripts/build_wttr_query.py', 'beijing'],
    ['python', '.agents/skills/weather-query-assistant/scripts/build_wttr_query.py', ' New   York ', '--mode', 'current'],
    ['python', '.agents/skills/weather-query-assistant/scripts/build_wttr_query.py', 'jfk', '--units', 'uscs'],
]

for cmd in examples:
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print('cmd =', ' '.join(cmd))
    print('out =', result.stdout.strip())
    print('returncode =', result.returncode)
    print('-' * 60)


cmd = python .agents/skills/weather-query-assistant/scripts/build_wttr_query.py beijing
out = https://wttr.in/Beijing?m&format=%l:+%c+%t+%h+%w
returncode = 0
------------------------------------------------------------
cmd = python .agents/skills/weather-query-assistant/scripts/build_wttr_query.py  New   York  --mode current
out = https://wttr.in/New+York?m&format=3
returncode = 0
------------------------------------------------------------
cmd = python .agents/skills/weather-query-assistant/scripts/build_wttr_query.py jfk --units uscs
out = https://wttr.in/JFK?u&format=%l:+%c+%t+%h+%w
returncode = 0
------------------------------------------------------------


## 把两个脚本串起来理解

现在这份天气 Skill 里已经有两个小脚本：

1. `normalize_location.py`
2. `build_wttr_query.py`

从职责上看：

- 第一个脚本负责清洗输入
- 第二个脚本负责拼出最终查询

这就是一个非常典型的“小而清晰的流程拆分”。


In [7]:
script_roles = {
    'normalize_location.py': 'clean and normalize raw user input',
    'build_wttr_query.py': 'turn normalized input into a query-ready wttr.in URL',
}

pprint(script_roles)


{'build_wttr_query.py': 'turn normalized input into a query-ready wttr.in URL',
 'normalize_location.py': 'clean and normalize raw user input'}


## 为什么不把这两步合成一个大脚本

这是正式开发里很重要的判断。

如果一上来就把所有东西塞进一个脚本，短期看起来快，长期会很难复用。

现在拆成两步的好处是：

- `normalize_location.py` 以后也可以被 Open-Meteo 查询复用
- `build_wttr_query.py` 可以专心处理 `wttr.in` 构建逻辑

也就是说，这种拆法更利于后续扩展。


In [8]:
why_not_one_big_script = [
    'normalization may be reused by other weather sources',
    'query building is source-specific',
    'smaller scripts are easier to test and replace',
]

pprint(why_not_one_big_script)


['normalization may be reused by other weather sources',
 'query building is source-specific',
 'smaller scripts are easier to test and replace']


## 这一步和真实使用的关系

现在如果用户问：

- “北京今天什么天气？”
- “帮我看下 New York 当前天气”

这份 Skill 的内部工作流已经可以很清楚地分成：

1. 判断地点是否明确
2. 标准化地点
3. 生成查询 URL
4. 再由查询能力实际取天气结果
5. 最后整理回答

也就是说，Skill 的结构已经开始真正支撑实际执行链路。


In [9]:
mini_workflow = [
    'check location clarity',
    'normalize location',
    'build wttr.in query URL',
    'fetch weather data',
    'answer concisely',
]

pprint(mini_workflow)


['check location clarity',
 'normalize location',
 'build wttr.in query URL',
 'fetch weather data',
 'answer concisely']


## 什么时候算“从脚本升级到流程”

一个很实用的判断标准是：

- 如果脚本之间开始有明确顺序关系
- 如果前一个脚本的输出成为后一个脚本的输入
- 如果这些动作共同支撑一个业务目标

那你就已经不只是在“加脚本”，而是在“搭流程”。


In [10]:
workflow_signals = {
    'ordered_steps': True,
    'output_to_input_chain': True,
    'shared_business_goal': True,
}

pprint(workflow_signals)


{'ordered_steps': True,
 'output_to_input_chain': True,
 'shared_business_goal': True}


## 这节课的正式开发价值

第五课最重要的不是多了一个脚本，而是你开始看到一件事：

- Skill 不只是文件集合
- 它会逐步长成一个有内在顺序的工作流

这也是为什么前面几课一直强调“按需增长”，而不是一开始就堆完整目录树。


## 如果继续往下走，第六课最自然的方向是什么

现在这份天气 Skill 已经有了：

- workflow
- metadata
- references
- scripts
- mini workflow

下一步最自然就是：

- 做一次完整演练

例如第六课可以做：

1. 输入一个用户问题
2. 判断地点是否需要澄清
3. 如果明确，则调用脚本生成最终查询 URL
4. 再给出适合执行的最终命令

这会让它更接近一次真实使用场景。


In [11]:
lesson_six_candidate = {
    'theme': 'run the weather skill as a full example session',
    'steps': [
        'read user question',
        'decide whether clarification is needed',
        'normalize location',
        'build final query URL or command',
        'prepare final answer pattern',
    ],
}

pprint(lesson_six_candidate)


{'steps': ['read user question',
           'decide whether clarification is needed',
           'normalize location',
           'build final query URL or command',
           'prepare final answer pattern'],
 'theme': 'run the weather skill as a full example session'}


## 当前阶段结论

你现在需要记住：

1. 一个 Skill 开始有多个小脚本协作时，就进入了 mini workflow 阶段
2. `normalize_location.py` 和 `build_wttr_query.py` 分别处理不同边界，拆分是有价值的
3. 好的 Skill 演进不是“越堆越大”，而是“每次增加一个合理的新层次”
4. 这份天气 Skill 现在已经具备了从输入到查询构建的最小流程能力
5. Skill 的成长顺序已经完整走到：最小版 -> metadata -> references -> scripts -> mini workflow

下一步建议：

- 继续第六课：做一次完整天气查询会话演练
